In [1]:
%matplotlib notebook
from rfsoc_rfdc.rfsoc_overlay import RFSoCOverlay
from rfsoc_rfdc.overlay_task import OverlayTask
from rfsoc_rfdc.overlay_task import BlinkLedTask

from rfsoc_rfdc.transmitter.single_ch_tx_task import SingleChTxTask
from rfsoc_rfdc.receiver.single_ch_rx_task import SingleChRxTask

from rfsoc_rfdc.transmitter.multi_ch_tx_mimo_task import MultiChTxMIMOTask
from rfsoc_rfdc.receiver.multi_ch_rx_mimo_task import MultiChRxMIMOTask

from rfsoc_rfdc.rfdc_task import RfdcTask 
from rfsoc_rfdc.cmac_task import CmacTask

from rfsoc_rfdc.rfdc_config import ZCU216_CONFIG

import sys
import os
import time

In [2]:
from rfsoc_rfdc.dsp.ofdm import OFDM
from rfsoc_rfdc.dsp.mimo_detection import MIMODetection

In [3]:
ol = RFSoCOverlay(path_to_bitstream="./rfsoc_rfdc/bitstream/rfsoc_rfdc_v45_rt_4ch_mts.bit")
NEW_CONFIG = {
    "RefClockForPLL": 300.0,
    "DACSampleRate": 2400.0,
    "DACInterpolationRate": 8,
    "DACNCO": 800,
    "ADCSampleRate": 2400.0,
    "ADCInterpolationRate": 8,
    "ADCNCO": -800                                           
}
ZCU216_CONFIG.update(NEW_CONFIG)

MTS_block/MTS_clk_wiz
adc_datapath/data_mover_ctrl
adc_datapath/fifo_count
axi_gpio_led
axi_intp
dac_datapath/data_mover_ctrl
dac_datapath/fifo_count
rfdc/usp_rf_data_converter
zynq_ultra_ps_e


In [4]:
true_samp_rate = ZCU216_CONFIG['DACSampleRate'] / ZCU216_CONFIG['DACInterpolationRate'] * 1e6

In [5]:
rfdc_t = RfdcTask(ol, mts_enable=True, debug_mode=True, board="ZCU216")

for task in [rfdc_t]:
    task.start()
    task.join()

2025-05-31 20:31:17,078 - root - WARNING - DAC NCO shall fall between -1/2*Fs and 1/2*Fs
2025-05-31 20:31:17,080 - root - WARNING - ADC NCO shall fall between 0 and 1/2*Fs
2025-05-31 20:31:17,302 - root - INFO - DAC tile 0 DAC block 0 is enabled!
2025-05-31 20:31:17,343 - root - INFO - DAC tile 0 DAC block 1 is enabled!
2025-05-31 20:31:17,383 - root - INFO - DAC tile 0 DAC block 2 is enabled!
2025-05-31 20:31:17,422 - root - INFO - DAC tile 0 DAC block 3 is enabled!
2025-05-31 20:31:17,462 - root - INFO - DAC tile 1 DAC block 0 is enabled!
2025-05-31 20:31:17,501 - root - INFO - DAC tile 1 DAC block 1 is enabled!
2025-05-31 20:31:17,541 - root - INFO - DAC tile 1 DAC block 2 is enabled!
2025-05-31 20:31:17,592 - root - INFO - DAC tile 1 DAC block 3 is enabled!
2025-05-31 20:31:17,632 - root - INFO - DAC tile 2 DAC block 0 is enabled!
2025-05-31 20:31:17,672 - root - INFO - DAC tile 2 DAC block 1 is enabled!
2025-05-31 20:31:17,711 - root - INFO - DAC tile 2 DAC block 2 is enabled!
202

In [6]:
for mcs in range(8):

    for atten in range(0, 31, 3):

        ZCU216_CONFIG['OFDM_ATTEN_DB'] = atten
        ZCU216_CONFIG['DETECTION_SCHEME'] = MIMODetection(sample_rate=true_samp_rate, tx_num=4, rx_num=4, MCS=mcs)
        ZCU216_CONFIG['CONFIG_NAME'] = f"Atten{atten}"

        print(f"MCS={mcs}, OFDM_ATTEN_DB={atten}")

        led_t = BlinkLedTask(ol)
        tx_t = MultiChTxMIMOTask(ol, mode="iq2real", channel_count=4, dp_vect_dim=1)
        rx_t = MultiChRxMIMOTask(ol, mode="real2iq", channel_count=4, dp_vect_dim=1)

        parallel_task = [led_t, tx_t, rx_t]
        for task in parallel_task:
            task.start()

        time.sleep(20)

        for task in parallel_task:
            task.stop()

MCS=0, OFDM_ATTEN_DB=0


FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

...

FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

2025-05-31 21:07:37,937 - root - INFO - Tx waveform waveTx_4_7.mat loaded successfully.
2025-05-31 21:07:38,259 - root - INFO - Tx data preparation done.
2025-05-31 21:07:41,417 - root - INFO - Rx samples captured.
2025-05-31 21:07:44,833 - root - WARNING - Rx detection failed.
2025-05-31 21:07:45,562 - root - INFO - Rx samples captured.
2025-05-31 21:07:48,979 - root - WARNING - Rx detection failed.
2025-05-31 21:07:49,708 - root - INFO - Rx samples captured.
2025-05-31 21:07:59,757 - root - INFO - Rx #CH0 SNR: 9.348, CFO: 0.000
2025-05-31 21:07:59,759 - root - INFO - Rx #CH1 SNR: 11.334, CFO: 0.000
2025-05-31 21:07:59,763 - root - INFO - Rx #CH2 SNR: 13.549, CFO: 0.000
2025-05-31 21:07:59,766 - root - INFO - Rx #CH3 SNR: 8.661, CFO: 0.000
